# Hyperparameter Tuning for Deep Nets

Learning-rate schedules and hyperparameter search — implemented from scratch on a
digit-classification MLP, validated against PyTorch schedulers, and compared on equal
trial budgets.

We answer four concrete questions with numbers:
1. How do constant, step, exponential, and cosine learning-rate schedules differ?
2. Do our cosine schedule values match PyTorch's `CosineAnnealingLR`?
3. How sensitive is validation accuracy to a **fixed** learning rate on digits?
4. With 9 trials, does random search cover the learning-rate axis more effectively
   than a 3×3 grid — even when both find similar best scores?


In [1]:
import warnings
warnings.filterwarnings('ignore')

import itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR, ExponentialLR, StepLR
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
plt.rcParams['figure.dpi'] = 100

## 1. Building Blocks: Adam and a Digit MLP

We reuse the Adam optimizer from Topic 05 and train a small ReLU MLP on
`sklearn.datasets.load_digits` (64 features → hidden → 10 classes).
All hyperparameter trials use the **same** train/validation split (`random_state=0`).

In [2]:
def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None:
            self.m = {k: np.zeros_like(p) for k, p in params.items()}
            self.v = {k: np.zeros_like(p) for k, p in params.items()}
        self.t += 1
        for k in params:
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * grads[k] ** 2
            m_hat = self.m[k] / (1 - self.beta1 ** self.t)
            v_hat = self.v[k] / (1 - self.beta2 ** self.t)
            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class DigitMLP:
    def __init__(self, hidden=64, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(64, hidden) * np.sqrt(2.0 / 64)
        self.b1 = np.zeros(hidden)
        self.W2 = rng.randn(hidden, 10) * np.sqrt(2.0 / hidden)
        self.b2 = np.zeros(10)

    def params_dict(self):
        return {'W1': self.W1, 'b1': self.b1, 'W2': self.W2, 'b2': self.b2}

    def set_params(self, params):
        for k in params:
            setattr(self, k, params[k])

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.maximum(0, self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        return self.z2

    def loss_and_grads(self, X, y):
        n = X.shape[0]
        logits = self.forward(X)
        probs = softmax(logits)
        y_oh = np.zeros_like(probs)
        y_oh[np.arange(n), y] = 1.0
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dz2 = (probs - y_oh) / n
        gW2 = self.a1.T @ dz2
        gb2 = dz2.sum(axis=0)
        dz1 = (dz2 @ self.W2.T) * (self.z1 > 0)
        gW1 = X.T @ dz1
        gb1 = dz1.sum(axis=0)
        return loss, {'W1': gW1, 'b1': gb1, 'W2': gW2, 'b2': gb2}

    def accuracy(self, X, y):
        pred = np.argmax(self.forward(X), axis=1)
        return np.mean(pred == y)


digits = load_digits()
X = StandardScaler().fit_transform(digits.data.astype(np.float64))
y = digits.target
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print(f'Digits: {X_train.shape[0]} train / {X_val.shape[0]} val, {X.shape[1]} features')

Digits: 1437 train / 360 val, 64 features


## 2. Learning-Rate Schedules From Scratch

Topic 05 swept **constant** learning rates across optimizers. Here we change $\eta$ **during**
training:

| Schedule | Rule at epoch $t$ (of $T$) |
|---|---|
| **Constant** | $\eta_t = \eta_0$ |
| **Step** | $\eta_t = \eta_0 \cdot \gamma^{\lfloor t / s \rfloor}$ |
| **Exponential** | $\eta_t = \eta_0 \cdot \gamma^t$ |
| **Cosine** | $\eta_t = \eta_{\min} + \tfrac{1}{2}(\eta_0 - \eta_{\min})(1 + \cos(\pi t / T))$ |

High initial $\eta$ speeds early progress; decay helps fine-tune later without the
oscillations of a large constant rate.

In [3]:
def lr_at_epoch(schedule, eta0, epoch, total_epochs, **kwargs):
    if schedule == 'constant':
        return eta0
    if schedule == 'step':
        return eta0 * (kwargs['gamma'] ** (epoch // kwargs['step_size']))
    if schedule == 'exponential':
        return eta0 * (kwargs['gamma'] ** epoch)
    if schedule == 'cosine':
        eta_min = kwargs['eta_min']
        progress = epoch / max(total_epochs - 1, 1)
        return eta_min + 0.5 * (eta0 - eta_min) * (1 + np.cos(np.pi * progress))
    raise ValueError(schedule)


T = 600
eta0 = 0.02
epochs = np.arange(T)
curves = {
    'constant': [lr_at_epoch('constant', eta0, t, T) for t in epochs],
    'step (÷2 every 150)': [lr_at_epoch('step', eta0, t, T, step_size=150, gamma=0.5) for t in epochs],
    'exp (γ=0.995)': [lr_at_epoch('exponential', eta0, t, T, gamma=0.995) for t in epochs],
    'cosine': [lr_at_epoch('cosine', eta0, t, T, eta_min=1e-4) for t in epochs],
}

fig, ax = plt.subplots(figsize=(8, 4))
for label, ys in curves.items():
    ax.plot(epochs, ys, label=label)
ax.set_xlabel('epoch'); ax.set_ylabel('learning rate')
ax.set_title('Learning-rate schedules (η₀=0.02, T=600)')
ax.legend(); ax.grid(alpha=0.3)
plt.savefig('lr_schedules.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Validate Cosine Schedule Against PyTorch

PyTorch's `CosineAnnealingLR` updates the optimizer **after** each epoch. We compare
our closed-form curve to the scheduler's recorded learning rates.

In [4]:
def train_with_schedule(schedule, eta0, epochs, seed=0, track_every=50, **sched_kw):
    model = DigitMLP(hidden=64, seed=seed)
    opt = Adam(lr=eta0)
    history = []
    for ep in range(epochs):
        opt.lr = lr_at_epoch(schedule, eta0, ep, epochs, **sched_kw)
        params = model.params_dict()
        loss, grads = model.loss_and_grads(X_train, y_train)
        opt.step(params, grads)
        model.set_params(params)
        if ep % track_every == 0 or ep == epochs - 1:
            history.append((ep, model.accuracy(X_val, y_val), loss))
    return history


opt_pt = torch.optim.SGD([torch.nn.Parameter(torch.tensor(1.0))], lr=eta0)
sched_pt = CosineAnnealingLR(opt_pt, T_max=T, eta_min=1e-4)
py_lrs = []
for _ in range(T):
    py_lrs.append(opt_pt.param_groups[0]['lr'])
    opt_pt.step()
    sched_pt.step()

np_lrs = [lr_at_epoch('cosine', eta0, ep, T, eta_min=1e-4) for ep in range(T)]
max_diff = max(abs(a - b) for a, b in zip(np_lrs, py_lrs))
print(f'Max |numpy − PyTorch| on cosine schedule: {max_diff:.2e}')

Max |numpy − PyTorch| on cosine schedule: 3.02e-05


## 4. Learning-Rate Sensitivity (Constant $\eta$)

Before scheduling, find a reasonable constant rate. We train 400 epochs at each rate
and report **validation** accuracy — never tune on the test set.

In [5]:
test_lrs = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
sens_acc = []
for lr in test_lrs:
    hist = train_with_schedule('constant', lr, epochs=400, track_every=400)
    sens_acc.append(hist[-1][1])

print(f"{'lr':>8}  {'val acc':>8}")
for lr, acc in zip(test_lrs, sens_acc):
    print(f'{lr:8.4f}  {acc:8.3f}')

best_idx = int(np.argmax(sens_acc))
print(f"\nBest constant lr in sweep: {test_lrs[best_idx]} → val acc {sens_acc[best_idx]:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(test_lrs, sens_acc, '-o')
ax.set_xscale('log')
ax.set_xlabel('constant learning rate')
ax.set_ylabel('validation accuracy')
ax.set_title('LR sensitivity (400 epochs, Adam, hidden=64)')
ax.grid(alpha=0.3)
plt.savefig('lr_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()

      lr   val acc
  0.0010     0.972
  0.0030     0.975
  0.0100     0.978
  0.0300     0.975
  0.1000     0.967
  0.3000     0.953

Best constant lr in sweep: 0.01 → val acc 0.978


## 5. Schedules vs Constant at High $\eta_0$

With $\eta_0 = 0.1$ (above the sweet spot from Section 4), decay schedules can
reduce late-training noise. On this small digit task, final validation accuracies are
similar — the honest takeaway is that schedules matter most when training is long,
noisy, or uses a deliberately aggressive initial rate.

In [6]:
high_eta = 0.1
n_epochs = 800
sched_configs = [
    ('constant', {}),
    ('step', {'step_size': 200, 'gamma': 0.3}),
    ('exponential', {'gamma': 0.995}),
    ('cosine', {'eta_min': 1e-4}),
]

print(f"{'schedule':>14}  {'final val':>10}  {'val @ ep200':>12}")
schedule_histories = {}
for name, kw in sched_configs:
    hist = train_with_schedule(name, high_eta, n_epochs, track_every=20, **kw)
    schedule_histories[name] = hist
    final_va = hist[-1][1]
    va200 = next(acc for ep, acc, _ in hist if ep == 200)
    print(f'{name:>14}  {final_va:10.3f}  {va200:12.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
for name, _ in sched_configs:
    eps = [h[0] for h in schedule_histories[name]]
    vas = [h[1] for h in schedule_histories[name]]
    ax.plot(eps, vas, label=name)
ax.set_xlabel('epoch'); ax.set_ylabel('validation accuracy')
ax.set_title(f'Validation curves (η₀={high_eta}, 800 epochs)')
ax.legend(); ax.grid(alpha=0.3)
plt.savefig('schedule_val_curves.png', dpi=100, bbox_inches='tight')
plt.show()

      schedule   final val   val @ ep200


      constant       0.967         0.964


          step       0.967         0.964


   exponential       0.969         0.969


        cosine       0.964         0.964


## 6. Hyperparameter Search: Grid vs Random

Following Bergstra & Bengio (2012): when some hyperparameters matter much more than
others, **random search** explores the important axes more efficiently than an equal-sized
grid.

We search learning rate and hidden width with a **fixed budget of 9 trials**. Grid search
uses a 3×3 lattice (only **3 distinct learning rates**). Random search draws learning rates
**log-uniformly** in $[10^{-3}, 10^{-0.5}]$, giving 9 distinct rates.

In [7]:
def run_trial(lr, hidden, seed, epochs=400):
    model = DigitMLP(hidden=hidden, seed=seed)
    opt = Adam(lr=lr)
    for _ in range(epochs):
        params = model.params_dict()
        _, grads = model.loss_and_grads(X_train, y_train)
        opt.step(params, grads)
        model.set_params(params)
    return model.accuracy(X_val, y_val)


def hyperparameter_search(mode, n_trials=9, seed=0, epochs=400):
    rng = np.random.RandomState(seed)
    if mode == 'grid':
        lrs = [0.001, 0.003, 0.01]
        hiddens = [32, 64, 128]
        configs = [{'lr': lr, 'hidden': h} for lr, h in itertools.product(lrs, hiddens)][:n_trials]
    else:
        log_lrs = [10 ** rng.uniform(-3.0, -0.5) for _ in range(n_trials)]
        hiddens = [16, 32, 64, 128, 256]
        configs = [{'lr': lr, 'hidden': int(rng.choice(hiddens))} for lr in log_lrs]

    records = []
    for i, cfg in enumerate(configs):
        va = run_trial(cfg['lr'], cfg['hidden'], seed=seed + i, epochs=epochs)
        records.append((cfg, va))
    best_cfg, best_va = max(records, key=lambda x: x[1])
    unique_lrs = len({round(r[0]['lr'], 8) for r in records})
    return best_cfg, best_va, unique_lrs, records


grid_cfg, grid_va, grid_ul, grid_recs = hyperparameter_search('grid')
rand_cfg, rand_va, rand_ul, rand_recs = hyperparameter_search('random')

print('=== 9-trial search (400 epochs per trial) ===')
print(f"Grid   best val={grid_va:.3f}  cfg={grid_cfg}  unique lrs={grid_ul}")
print(f"Random best val={rand_va:.3f}  cfg={rand_cfg}  unique lrs={rand_ul}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for ax_i, (recs, title) in zip(ax, [(grid_recs, 'grid'), (rand_recs, 'random')]):
    lrs = [r[0]['lr'] for r in recs]
    vas = [r[1] for r in recs]
    sc = ax_i.scatter(lrs, vas, c=vas, cmap='viridis', s=80)
    ax_i.set_xscale('log')
    ax_i.set_xlabel('learning rate')
    ax_i.set_ylabel('val accuracy')
    ax_i.set_title(f'{title} search trials')
    ax_i.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('search_trials.png', dpi=100, bbox_inches='tight')
plt.show()

=== 9-trial search (400 epochs per trial) ===
Grid   best val=0.981  cfg={'lr': 0.003, 'hidden': 128}  unique lrs=3
Random best val=0.981  cfg={'lr': 0.02355232360424789, 'hidden': 64}  unique lrs=9


## 7. Early Stopping as Implicit Tuning

Topic 07 treated early stopping as regularization. It is also a hyperparameter strategy:
the **patience** and **metric** you monitor are knobs, and stopping early can mimic
annealing a learning rate when validation loss plateaus.

```python
# Pattern (not re-run here — see Topic 07):
best_val, wait = -np.inf, 0
for epoch in range(max_epochs):
    train_one_epoch(...)
    val_score = evaluate_val(...)
    if val_score > best_val:
        best_val, wait = val_score, 0
        save_checkpoint()
    else:
        wait += 1
        if wait >= patience:
            break
```

Combine explicit LR schedules with early stopping on validation loss for a practical
default recipe.

## Summary

- **Schedules** let you start fast and finish fine; formulas match PyTorch within ~3e-5.
- **Constant LR sweeps** on digits peak near $\eta=0.01$ (~0.978 val acc); $\eta=0.3$ drops to ~0.953.
- At $\eta_0=0.1$, schedule choice barely changes final val acc on this easy task — still
  plot the curves and validate the math.
- **Random search** with 9 trials tries **9 distinct learning rates** vs **3** for a 3×3 grid,
  covering the sensitive dimension more thoroughly even when best scores tie (~0.981).
- Always tune on **validation**, report variance across seeds, and never cherry-pick the
  best of many trials without saying so.